<a href="https://colab.research.google.com/github/caiopsd/video-classifier/blob/main/genvidbench_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GenVidBench (Pair1) - Treinamento Temporal Real vs Fake

Notebook para carregar o dataset a partir do Google Drive, fazer pre-processamento temporal e treinar um modelo que usa **sequencia de frames** (dinamica temporal), nao apenas aparencia de imagem.

Objetivo deste notebook:
- Ler `Pair1_labels.txt` em `My Drive/GenVidBench/GenVidBench`
- Usar `hf_label` como alvo (`1 = fake`, `0 = real`)
- Pre-processar clipes de video (amostragem temporal + normalizacao)
- Treinar modelo com encoder por frame + GRU temporal
- Avaliar com acuracia, F1 e matriz de confusao


In [1]:
# Se estiver no Colab, monte o Drive:
from google.colab import drive
drive.mount('/content/drive')

# Dependencias opcionais (descomente no Colab se necessario)
!pip install -q decord==0.6.0

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 129.6 MB/s eta 0:00:00


In [2]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

try:
    import decord
    from decord import VideoReader, cpu
    HAS_DECORD = True
except Exception:
    HAS_DECORD = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Decord disponivel:', HAS_DECORD)

Device: cuda
Decord disponivel: True


In [3]:
# Caminho informado por voce
DATASET_ROOT = Path('/content/drive/MyDrive/GenVidBench/GenVidBench')
PAIR1_DIR = DATASET_ROOT / 'Pair1'
LABEL_FILE = DATASET_ROOT / 'Pair1_labels.txt'

print('DATASET_ROOT exists:', DATASET_ROOT.exists())
print('PAIR1_DIR exists:', PAIR1_DIR.exists())
print('LABEL_FILE exists:', LABEL_FILE.exists())

if not LABEL_FILE.exists():
    raise FileNotFoundError(f'Nao encontrei {LABEL_FILE}')

DATASET_ROOT exists: True
PAIR1_DIR exists: True
LABEL_FILE exists: True


In [4]:
# Leitura de labels
rows = []
with LABEL_FILE.open('r', encoding='utf-8') as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            rel_path, hf_label = line.rsplit(' ', 1)
            hf_label = int(hf_label)
        except Exception:
            continue

        rel_path = rel_path.replace('\\', '/')
        abs_path = DATASET_ROOT / rel_path

        model_folder = Path(rel_path).parts[1] if len(Path(rel_path).parts) > 1 else 'unknown'

        # Regra definida: hf_label == 1 -> fake, hf_label == 0 -> real
        if hf_label not in (0, 1):
            continue

        rows.append({
            'rel_path': rel_path,
            'abs_path': str(abs_path),
            'hf_label': hf_label,
            'source_folder': model_folder,
            'exists': abs_path.exists(),
        })

df = pd.DataFrame(rows)
df = df[df['exists']].reset_index(drop=True)
print('Total de videos validos:', len(df))
df.head()


Total de videos validos: 74135


,rel_path,abs_path,hf_label,source_folder,exists
0,Pair1/ms/ms-0015c714-5c20-5766-8944-540102ec09...,/content/drive/MyDrive/GenVidBench/GenVidBench...,1,ms,True
1,Pair1/ms/ms-001c5b92-f5c3-577d-a80f-cde08a9ef1...,/content/drive/MyDrive/GenVidBench/GenVidBench...,1,ms,True
2,Pair1/ms/ms-002a629a-e7db-51c0-8613-cb5ab2cbf7...,/content/drive/MyDrive/GenVidBench/GenVidBench...,1,ms,True
3,Pair1/ms/ms-002a6bed-74b8-547b-bf58-f7abd5fd67...,/content/drive/MyDrive/GenVidBench/GenVidBench...,1,ms,True
4,Pair1/ms/ms-002acde4-7101-5176-b1e1-fb9ba768b7...,/content/drive/MyDrive/GenVidBench/GenVidBench...,1,ms,True


In [5]:
print('Distribuicao por hf_label (0=real, 1=fake):')
print(df['hf_label'].value_counts(dropna=False))

print('\nDistribuicao por pasta de origem:')
print(df['source_folder'].value_counts().head(10))


Distribuicao por hf_label (0=real, 1=fake):
hf_label
1    54004
0    20131
Name: count, dtype: int64

Distribuicao por pasta de origem:
source_folder
vript    20131
ms       13501
pika     13501
t2vz     13501
vc2      13501
Name: count, dtype: int64


In [6]:
# Opcional: reduzir para experimento rapido
MAX_PER_CLASS = 2500  # ajuste para caber no seu tempo/GPU

parts = []
for y in [0, 1]:
    chunk = df[df['hf_label'] == y]
    if len(chunk) > MAX_PER_CLASS:
        chunk = chunk.sample(MAX_PER_CLASS, random_state=SEED)
    parts.append(chunk)
df_trainable = pd.concat(parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print('Total usado para treino/valid/teste:', len(df_trainable))
print(df_trainable['hf_label'].value_counts())


Total usado para treino/valid/teste: 5000
hf_label
0    2500
1    2500
Name: count, dtype: int64


In [7]:
train_df, temp_df = train_test_split(
    df_trainable,
    test_size=0.30,
    stratify=df_trainable['hf_label'],
    random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['hf_label'],
    random_state=SEED,
)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')


Train: 3500 | Val: 750 | Test: 750


In [8]:
class Pair1TemporalDataset(Dataset):
    def __init__(self, frame_df, num_frames=16, image_size=160):
        self.df = frame_df.reset_index(drop=True)
        self.num_frames = num_frames
        self.image_size = image_size
        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.df)

    def _sample_indices(self, total):
        if total <= 0:
            return np.zeros(self.num_frames, dtype=np.int64)
        if total >= self.num_frames:
            return np.linspace(0, total - 1, self.num_frames).astype(np.int64)
        idx = np.linspace(0, total - 1, total).astype(np.int64).tolist()
        while len(idx) < self.num_frames:
            idx.append(idx[-1])
        return np.array(idx, dtype=np.int64)

    def _read_video_decord(self, path):
        vr = VideoReader(path, ctx=cpu(0))
        idx = self._sample_indices(len(vr))
        frames = vr.get_batch(idx).asnumpy()  # T,H,W,C
        return frames

    def _read_video_cv2(self, path):
        import cv2
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        idx = set(self._sample_indices(total).tolist())

        frames = []
        i = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if i in idx:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
            i += 1
        cap.release()

        if len(frames) == 0:
            raise RuntimeError('Falha ao ler frames via cv2')

        while len(frames) < self.num_frames:
            frames.append(frames[-1])
        return np.stack(frames[: self.num_frames], axis=0)

    def _to_tensor_clip(self, frames_np):
        # T,H,W,C -> T,C,H,W -> redimensiona frame a frame
        x = torch.from_numpy(frames_np).float() / 255.0
        x = x.permute(0, 3, 1, 2)

        resized = []
        for t in range(x.shape[0]):
            fr = x[t]
            fr = T.functional.resize(fr, [self.image_size, self.image_size], antialias=True)
            fr = self.normalize(fr)
            resized.append(fr)

        x = torch.stack(resized, dim=0)  # T,C,H,W
        x = x.permute(1, 0, 2, 3)        # C,T,H,W
        return x

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_path = row['abs_path']

        try:
            if HAS_DECORD:
                frames = self._read_video_decord(video_path)
            else:
                frames = self._read_video_cv2(video_path)
            clip = self._to_tensor_clip(frames)
        except Exception:
            # fallback para evitar quebrar treino por 1 video corrompido
            clip = torch.zeros(3, self.num_frames, self.image_size, self.image_size)

        label = torch.tensor(float(row['hf_label']), dtype=torch.float32)
        return clip, label


In [9]:
NUM_FRAMES = 16
IMG_SIZE = 160
BATCH_SIZE = 6 if DEVICE == 'cuda' else 2

train_ds = Pair1TemporalDataset(train_df, num_frames=NUM_FRAMES, image_size=IMG_SIZE)
val_ds = Pair1TemporalDataset(val_df, num_frames=NUM_FRAMES, image_size=IMG_SIZE)
test_ds = Pair1TemporalDataset(test_df, num_frames=NUM_FRAMES, image_size=IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

x, y = next(iter(train_loader))
print('Batch video shape [B,C,T,H,W]:', x.shape)
print('Batch labels shape:', y.shape)

Batch video shape [B,C,T,H,W]: torch.Size([6, 3, 16, 160, 160])
Batch labels shape: torch.Size([6])


In [10]:
class TemporalGRUClassifier(nn.Module):
    """
    Encoder por frame (ResNet18) + GRU temporal.
    Inclui sinal de movimento com diferenca entre embeddings consecutivos.
    """
    def __init__(self, hidden_size=256, dropout=0.2):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])  # [B,512,1,1]
        self.feature_dim = 512

        self.gru = nn.GRU(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_size * 2 + 1, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        # x: [B,C,T,H,W]
        b, c, t, h, w = x.shape
        frames = x.permute(0, 2, 1, 3, 4).reshape(b * t, c, h, w)

        feat = self.encoder(frames).flatten(1)       # [B*T, 512]
        feat = feat.reshape(b, t, self.feature_dim)  # [B, T, 512]

        # Sinal explicito de movimento (delta entre embeddings temporais)
        if t > 1:
            delta = feat[:, 1:, :] - feat[:, :-1, :]
            motion = delta.abs().mean(dim=(1, 2), keepdim=False).unsqueeze(1)  # [B,1]
        else:
            motion = torch.zeros((b, 1), device=x.device)

        out, _ = self.gru(feat)
        temporal_embed = out[:, -1, :]

        fused = torch.cat([temporal_embed, motion], dim=1)
        logits = self.head(fused).squeeze(1)
        return logits

model = TemporalGRUClassifier().to(DEVICE)
print('Modelo criado.')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 240MB/s]


Modelo criado.


In [11]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    criterion = nn.BCEWithLogitsLoss()
    losses = []
    y_true = []
    y_pred = []

    for clips, labels in loader:
        clips = clips.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            logits = model(clips)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()

        losses.append(loss.item())
        y_true.extend(labels.detach().cpu().long().tolist())
        y_pred.extend(preds.detach().cpu().tolist())

    avg_loss = float(np.mean(losses)) if losses else 0.0
    acc = accuracy_score(y_true, y_pred) if y_true else 0.0
    f1 = f1_score(y_true, y_pred, zero_division=0) if y_true else 0.0
    return avg_loss, acc, f1

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
EPOCHS = 5

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_f1 = run_epoch(model, train_loader, optimizer=optimizer)
    va_loss, va_acc, va_f1 = run_epoch(model, val_loader, optimizer=None)

    print(
        f'Epoch {epoch:02d} | '
        f'train_loss={tr_loss:.4f} acc={tr_acc:.4f} f1={tr_f1:.4f} | '
        f'val_loss={va_loss:.4f} acc={va_acc:.4f} f1={va_f1:.4f}'
    )

torch.save(model.state_dict(), 'temporal_pair1_real_fake.pt')
print('Pesos salvos em temporal_pair1_real_fake.pt')

Epoch 01 | train_loss=0.4465 acc=0.8109 f1=0.8081 | val_loss=0.2284 acc=0.9067 f1=0.9006
Epoch 02 | train_loss=0.3317 acc=0.8606 f1=0.8587 | val_loss=0.2356 acc=0.9053 f1=0.9047
Epoch 03 | train_loss=0.2881 acc=0.8911 f1=0.8901 | val_loss=0.2885 acc=0.9040 f1=0.9104
Epoch 04 | train_loss=0.2619 acc=0.8946 f1=0.8942 | val_loss=0.1858 acc=0.9360 f1=0.9358
Epoch 05 | train_loss=0.2350 acc=0.9109 f1=0.9104 | val_loss=0.2083 acc=0.9253 f1=0.9296
Pesos salvos em temporal_pair1_real_fake.pt


In [12]:
# Avaliacao final
model.eval()
all_true = []
all_pred = []

with torch.no_grad():
    for clips, labels in test_loader:
        clips = clips.to(DEVICE)
        logits = model(clips)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long().cpu().numpy()

        all_pred.extend(preds.tolist())
        all_true.extend(labels.long().numpy().tolist())

print('Test accuracy:', accuracy_score(all_true, all_pred))
print('Test F1:', f1_score(all_true, all_pred, zero_division=0))
print('\nClassification report:')
print(classification_report(all_true, all_pred, target_names=['real', 'fake'], zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(all_true, all_pred))


Test accuracy: 0.9306666666666666
Test F1: 0.9343434343434344

Classification report:
              precision    recall  f1-score   support

        real       0.98      0.87      0.93       375
        fake       0.89      0.99      0.93       375

    accuracy                           0.93       750
   macro avg       0.94      0.93      0.93       750
weighted avg       0.94      0.93      0.93       750

Confusion matrix:
[[328  47]
 [  5 370]]


## Nota Importante

Este notebook usa **dinamica temporal** por duas vias:
- sequencia de embeddings por frame processada por GRU
- sinal de movimento por diferenca entre embeddings consecutivos

Assim o modelo nao depende apenas de textura/aparencia estatica de uma unica imagem.